In [1]:
# this code will take the adoption model results that are the change in stock counts for each scenario compared to baseline
# and convert those to dollar savings based on avoided cost tables that Mike will provide

In [2]:
# eventually we will need to make this dynamic based on the cost test selected
# and have Mike create the various tables for avoided cost that the measure will carry with them
# but for now I will just make 10 copies of the 2026 results to use

#(Avoided cost Needs to be dependent on the cost test (Real discount rate will change)
#  now just an option but will keep real discount rate as 0.03 for all cost tests for now)

#the avoided cost tables are Cost by widget and year 

# When a widget is created the savings and costs for the entire lifetime are reported in that year 
# (the cost and savings are already discounted over the life time in the avoided costs inputs)

In [3]:
import pandas as pd
import numpy as np

In [4]:
# this code reads in the adoption model results for technical potential compared to baseline
competition_results_df = pd.read_pickle("050_input/competition_diff_adoption_results.pkl")
tech_results_df = pd.read_pickle("050_input/tech_diff_adoption_results.pkl")
# we Just care about the measures created
# the avoided costs metrics are at the measure level so the baseline energy is already removed from that calculation
# So we will drop the baseline and existing columns as we dont care what value they are
# but we do care about the efficient and top10 columns because they tell use where each equipment came from (stock and what it has become)

In [5]:
tech_results_df

,competition_group,electric_utility,gas_utility,building_type,time,remaining_below_baseline_stock,remaining_baseline_stock,remaining_efficient_stock,remaining_top10_stock,ret_er_below_baseline_stock,ret_er_baseline_stock,ret_er_efficient_stock,ret_er_top10_stock,rob_below_baseline_stock,rob_baseline_stock,rob_efficient_stock,rob_top10_stock
0,refrigeration,test_utility_1,test_utility_1,multifamily,0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000
1,refrigeration,test_utility_1,test_utility_1,multifamily,1,0.0,-3000.000000,0.0,3000.000000,0.0,-120.000000,0.0,120.000000,0.0,-60.000000,0.0,60.000000
2,refrigeration,test_utility_1,test_utility_1,multifamily,2,0.0,-5500.000000,0.0,5500.000000,0.0,-698.400000,0.0,698.400000,0.0,-99.200000,0.0,99.200000
3,refrigeration,test_utility_1,test_utility_1,multifamily,3,0.0,-7583.333333,0.0,7583.333333,0.0,-1492.954667,0.0,1492.954667,0.0,-288.144000,0.0,288.144000
4,refrigeration,test_utility_1,test_utility_1,multifamily,4,0.0,-9319.444444,0.0,9319.444444,0.0,-2359.285938,0.0,2359.285938,0.0,-547.698524,0.0,547.698524
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,heating_cooling,test_utility_2,test_utility_2,single_family_li,15,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000
1436,heating_cooling,test_utility_2,test_utility_2,single_family_li,16,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000
1437,heating_cooling,test_utility_2,test_utility_2,single_family_li,17,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000
1438,heating_cooling,test_utility_2,test_utility_2,single_family_li,18,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000


In [6]:
# this code drops the unneeded columns from tech_results_df any column with _baseline in it
columns_to_drop = [col for col in tech_results_df.columns if '_baseline' in col or '_below_baseline' in col]
tech_results_df = tech_results_df.drop(columns=columns_to_drop)

# Do the same for competition_results_df
columns_to_drop = [col for col in competition_results_df.columns if '_baseline' in col or '_below_baseline' in col]
competition_results_df = competition_results_df.drop(columns=columns_to_drop)

tech_results_df

,competition_group,electric_utility,gas_utility,building_type,time,remaining_efficient_stock,remaining_top10_stock,ret_er_efficient_stock,ret_er_top10_stock,rob_efficient_stock,rob_top10_stock
0,refrigeration,test_utility_1,test_utility_1,multifamily,0,0.0,0.000000,0.0,0.000000,0.0,0.000000
1,refrigeration,test_utility_1,test_utility_1,multifamily,1,0.0,3000.000000,0.0,120.000000,0.0,60.000000
2,refrigeration,test_utility_1,test_utility_1,multifamily,2,0.0,5500.000000,0.0,698.400000,0.0,99.200000
3,refrigeration,test_utility_1,test_utility_1,multifamily,3,0.0,7583.333333,0.0,1492.954667,0.0,288.144000
4,refrigeration,test_utility_1,test_utility_1,multifamily,4,0.0,9319.444444,0.0,2359.285938,0.0,547.698524
...,...,...,...,...,...,...,...,...,...,...,...
1435,heating_cooling,test_utility_2,test_utility_2,single_family_li,15,0.0,0.000000,0.0,0.000000,0.0,0.000000
1436,heating_cooling,test_utility_2,test_utility_2,single_family_li,16,0.0,0.000000,0.0,0.000000,0.0,0.000000
1437,heating_cooling,test_utility_2,test_utility_2,single_family_li,17,0.0,0.000000,0.0,0.000000,0.0,0.000000
1438,heating_cooling,test_utility_2,test_utility_2,single_family_li,18,0.0,0.000000,0.0,0.000000,0.0,0.000000


In [7]:
#this code calculates the change in stocks over time. It looks at the time column and calculated each row value example 1-2 
# Compute per-step adoptions (differences) on the long-form data for all dataframes
id_cols = ['competition_group', 'electric_utility', 'gas_utility', 'building_type', 'market', 'efficiency_level']

# Store results for both tech and competition
adoption_results = {}

# Process both dataframes
processed_dfs = {
    'tech': tech_results_df,
    'competition': competition_results_df
}

for name, results_df in processed_dfs.items():
    print(f"\nComputing adoptions for {name}_results_df...")
    
    # Recreate df_long the same way as before
    import re
    pattern = r'^(ret_er|rob|remaining)_(efficient|top10)_stock$'
    stock_cols = [c for c in results_df.columns if re.match(pattern, c)]
    df_long = results_df.melt(
        id_vars=['competition_group', 'electric_utility', 'gas_utility', 'building_type', 'time'],
        value_vars=stock_cols,
        var_name='market_eff',
        value_name='value'
    )
    df_long[['market','efficiency_level']] = df_long['market_eff'].str.extract(pattern)
    df_long = df_long.drop(columns=['market_eff'])
    
    # Sort and compute diffs
    df_long = df_long.sort_values(id_cols + ['time'])
    
    grouped = df_long.groupby(id_cols)['value']
    # Backward diff: value - previous value (NaN for first time)
    df_long['adoption'] = grouped.diff().fillna(0)
    
    # Pivot adoption (backward diff) to wide
    df_adopt_wide = df_long.pivot_table(
        index=id_cols,
        columns='time',
        values='adoption',
        aggfunc='first'
    ).reset_index()

    
    # Rename time columns to adopt_t_<time>
    def rename_time_cols(df, prefix='adopt_t'):
        df.columns.name = None
        new_cols = []
        for c in df.columns:
            if isinstance(c, (int, np.integer, float)):
                new_cols.append(f"{prefix}_{c}")
            else:
                new_cols.append(c)
        df.columns = new_cols
        return df
    
    df_adopt_wide = rename_time_cols(df_adopt_wide, prefix='adopt_t')
    
    print(f'Backward-diff adoption shape: {df_adopt_wide.shape}')
    
    adoption_results[name] = {
        'adopt_wide': df_adopt_wide
    }

# Display sample of tech results
print("\nTech adoption sample:")
display(adoption_results['tech']['adopt_wide'])



Computing adoptions for tech_results_df...
Backward-diff adoption shape: (432, 26)

Computing adoptions for competition_results_df...
Backward-diff adoption shape: (432, 26)

Tech adoption sample:


,competition_group,electric_utility,gas_utility,building_type,market,efficiency_level,adopt_t_0,adopt_t_1,adopt_t_2,adopt_t_3,...,adopt_t_10,adopt_t_11,adopt_t_12,adopt_t_13,adopt_t_14,adopt_t_15,adopt_t_16,adopt_t_17,adopt_t_18,adopt_t_19
0,heating_cooling,none,none,multifamily,remaining,efficient,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,heating_cooling,none,none,multifamily,remaining,top10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,heating_cooling,none,none,multifamily,ret_er,efficient,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,heating_cooling,none,none,multifamily,ret_er,top10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,heating_cooling,none,none,multifamily,rob,efficient,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,refrigeration,test_utility_2,test_utility_2,single_family_li,remaining,top10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
428,refrigeration,test_utility_2,test_utility_2,single_family_li,ret_er,efficient,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
429,refrigeration,test_utility_2,test_utility_2,single_family_li,ret_er,top10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
430,refrigeration,test_utility_2,test_utility_2,single_family_li,rob,efficient,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Save all adoption results to pickle and CSV files
for name in adoption_results.keys():
    df_adopt = adoption_results[name]['adopt_wide']
    output_pkl = f"050_output/{name}_adoption_changes.pkl"
    output_csv = f"050_output/{name}_adoption_changes.csv"
    df_adopt.to_pickle(output_pkl)
    df_adopt.to_csv(output_csv, index=False)
    print(f"Saved {name} adoption changes to {output_pkl} and {output_csv}")

Saved tech adoption changes to 050_output/tech_adoption_changes.pkl and 050_output/tech_adoption_changes.csv
Saved competition adoption changes to 050_output/competition_adoption_changes.pkl and 050_output/competition_adoption_changes.csv


In [9]:
# now we just sum up the total created because the stock doesnt tell us where they came from just the age of the asset
# only works in first year
# now with the summed total we know 2 percent of the Ret_ER population for that year was upgraded and 100 % of the rob was upgraded
# the rob to ret_er proportions to 1/eul and (1- 1/eul) * (1/3) * (0.02)


In [10]:
# this code collapses the market column into a total stock column and sums up all the numeric values 
# in the adoption model results df
collapsed_adoption_results = {}

for name in adoption_results.keys():
    df_adopt = adoption_results[name]['adopt_wide']
    
    # Get all the adopt_t columns (numeric columns)
    adopt_cols = [col for col in df_adopt.columns if col.startswith('adopt_t_')]
    
    # Group by all id columns except 'market' and sum the adoption values
    id_cols_no_market = ['competition_group', 'electric_utility', 'gas_utility', 'building_type', 'efficiency_level']
    
    df_collapsed = df_adopt.groupby(id_cols_no_market, as_index=False)[adopt_cols].sum()
    
    print(f"\n{name.capitalize()} collapsed adoption shape: {df_collapsed.shape}")
    collapsed_adoption_results[name] = df_collapsed

# Display sample
print("\nTech collapsed adoption sample:")
display(collapsed_adoption_results['tech'])
print("\nCompetition collapsed adoption sample:")
display(collapsed_adoption_results['competition'])


Tech collapsed adoption shape: (144, 25)

Competition collapsed adoption shape: (144, 25)

Tech collapsed adoption sample:


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,adopt_t_0,adopt_t_1,adopt_t_2,adopt_t_3,adopt_t_4,...,adopt_t_10,adopt_t_11,adopt_t_12,adopt_t_13,adopt_t_14,adopt_t_15,adopt_t_16,adopt_t_17,adopt_t_18,adopt_t_19
0,heating_cooling,none,none,multifamily,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,heating_cooling,none,none,multifamily,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,heating_cooling,none,none,multifamily_li,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,heating_cooling,none,none,multifamily_li,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,heating_cooling,none,none,single_family,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
140,refrigeration,test_utility_2,test_utility_2,single_family,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
141,refrigeration,test_utility_2,test_utility_2,single_family,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
142,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Competition collapsed adoption sample:


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,adopt_t_0,adopt_t_1,adopt_t_2,adopt_t_3,adopt_t_4,...,adopt_t_10,adopt_t_11,adopt_t_12,adopt_t_13,adopt_t_14,adopt_t_15,adopt_t_16,adopt_t_17,adopt_t_18,adopt_t_19
0,heating_cooling,none,none,multifamily,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,heating_cooling,none,none,multifamily,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,heating_cooling,none,none,multifamily_li,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,heating_cooling,none,none,multifamily_li,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,heating_cooling,none,none,single_family,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
140,refrigeration,test_utility_2,test_utility_2,single_family,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
141,refrigeration,test_utility_2,test_utility_2,single_family,top10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
142,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# we know that the relationship between ret_er and rob adoptions 
# will always be the same as defined in the adoption model
# so we can split the collapsed adoption results back into ret_er and rob components 
# these components will then be used to calculate the dollar savings based on avoided cost tables

In [12]:
def clean_column_names(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('$', 'usd').str.replace('/', 'per').str.replace('&', 'and').str.replace('.', '')
    return df

In [13]:
#read in pickle file
# remember market is now an age thing not a replacement thing
df_yrs = pd.read_pickle("./030_input/df_yrs.pkl")
df_yrs
# filter df_years to remove any rows that have baseline or existing in the condition column
df_yrs = df_yrs[~df_yrs['condition'].str.contains('baseline|existing', case=False, na=False)]

# quickly make column efficiency_level based on the column condition_name. if condition_name contains 'efficient' then efficiency_level is 'efficient' else 'top10'
df_yrs['efficiency_level'] = np.where(df_yrs['condition'].str.contains('efficient', case=False, na=False), 'efficient', 'top10')
df_yrs = clean_column_names(df_yrs)
df_yrs

,condition,competition_group,subgroup,electric_utility,gas_utility,building_type,measure_life_(yrs),initial_count,market,count,...,sct_cost,sct_benefit,sct_bcr,rim_cost,rim_benefit,rim_bcr,pct_cost,pct_benefit,pct_bcr,efficiency_level
0,furnace_oil_efficient_residential,heating_cooling,oil_furnace,test_utility_1,test_utility_1,single_family,10,5000.0,ROB,500.0,...,1680.0,3636.660007,2.164679,1172.716654,4138.658036,3.52912,700.0,4682.630596,6.689472,efficient
1,furnace_oil_efficient_residential,heating_cooling,oil_furnace,test_utility_1,test_utility_2,single_family,10,10000.0,ROB,1000.0,...,1680.0,3645.190210,2.169756,1172.716654,4138.658036,3.52912,700.0,4691.160799,6.701658,efficient
2,furnace_oil_efficient_residential,heating_cooling,oil_furnace,test_utility_1,none,single_family,10,27500.0,ROB,2750.0,...,1680.0,3636.660007,2.164679,1172.716654,4138.658036,3.52912,700.0,4682.630596,6.689472,efficient
9,furnace_natural_gas_efficient_residential,heating_cooling,gas_furnace,test_utility_1,test_utility_1,single_family,10,5000.0,ROB,500.0,...,1680.0,3636.660007,2.164679,1172.716654,4138.658036,3.52912,700.0,4682.630596,6.689472,efficient
10,furnace_natural_gas_efficient_residential,heating_cooling,gas_furnace,test_utility_1,test_utility_2,single_family,10,10000.0,ROB,1000.0,...,1680.0,3645.190210,2.169756,1172.716654,4138.658036,3.52912,700.0,4691.160799,6.701658,efficient
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,refrigerator_electricity_efficient_residential,refrigeration,full_size,test_utility_1,test_utility_2,multifamily_li,10,3600.0,REMAINING,2160.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efficient
350,refrigerator_electricity_efficient_residential,refrigeration,full_size,test_utility_1,none,multifamily_li,10,2520.0,REMAINING,1512.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efficient
354,refrigerator_electricity_top10_residential,refrigeration,full_size,test_utility_1,test_utility_1,multifamily_li,10,750.0,REMAINING,450.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,top10
355,refrigerator_electricity_top10_residential,refrigeration,full_size,test_utility_1,test_utility_2,multifamily_li,10,3000.0,REMAINING,1800.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,top10


In [14]:
# make a unique list every competition group and thir EUL
unique_competition_groups = df_yrs[['competition_group', 'measure_life_(yrs)']].drop_duplicates()
# this might need to be expanded to include the subgroups

In [15]:
# join the unique competition groups and their EUL to the collapsed adoption results
# and use EUL in the split calculation total_stock cancels out so we dont need it
# Starting from total stock formulas:
#   rob_adopted = total_stock * (1/EUL)
#   ret_er_adopted = total_stock * (1 - 1/EUL) * (1/3) * 0.02
#   total_adopted = rob_adopted + ret_er_adopted
# We can derive the proportions to split total_adopted back into components
for name in collapsed_adoption_results.keys():
    df_collapsed = collapsed_adoption_results[name]
    # Merge with unique competition groups to get EUL
    df_collapsed = df_collapsed.merge(unique_competition_groups, on='competition_group', how='left')
    
    # Calculate the proportion factors
    # rob_factor = (1/EUL)
    # ret_er_factor = (1 - 1/EUL) * (1/3) * 0.02
    # total_factor = rob_factor + ret_er_factor (this is what total_adopted/total_stock equals)
    rob_factor = 1 / df_collapsed['measure_life_(yrs)']
    ret_er_factor = (1 - 1/df_collapsed['measure_life_(yrs)']) * (1/3) * 0.02
    total_factor = rob_factor + ret_er_factor
    
    # Split each adopt_t_<time> column into ret_er and rob components
    adopt_cols = [col for col in df_collapsed.columns if col.startswith('adopt_t_')]
    
    for col in adopt_cols:
        # ROB allocation: total_adopted * (rob_factor / total_factor)
        df_collapsed[f'rob_{col}'] = df_collapsed[col] * (rob_factor / total_factor)
        
        # RET_ER allocation: total_adopted * (ret_er_factor / total_factor)
        df_collapsed[f'ret_er_{col}'] = df_collapsed[col] * (ret_er_factor / total_factor)
    
    # Drop the original collapsed adopt_t_<time> columns
    df_collapsed = df_collapsed.drop(columns=adopt_cols)
    
    # Update the collapsed adoption results
    collapsed_adoption_results[name] = df_collapsed

    print(f"\n{name.capitalize()} split adoption shape: {df_collapsed.shape}")
df_collapsed


Tech split adoption shape: (144, 46)

Competition split adoption shape: (144, 46)


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs),rob_adopt_t_0,ret_er_adopt_t_0,rob_adopt_t_1,ret_er_adopt_t_1,...,rob_adopt_t_15,ret_er_adopt_t_15,rob_adopt_t_16,ret_er_adopt_t_16,rob_adopt_t_17,ret_er_adopt_t_17,rob_adopt_t_18,ret_er_adopt_t_18,rob_adopt_t_19,ret_er_adopt_t_19
0,heating_cooling,none,none,multifamily,efficient,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,heating_cooling,none,none,multifamily,top10,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,heating_cooling,none,none,multifamily_li,efficient,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,heating_cooling,none,none,multifamily_li,top10,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,heating_cooling,none,none,single_family,efficient,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
140,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
141,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
142,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# this code takes the df_collapsed and splits it back into ret_er and rob components for every time column
# the time columns are currently called adopt_t_<time>
# the new columns will be called ret_er_adopt_t_<time> and rob_adopt_t_<time>
# 
# Allocation logic (derived from total stock relationships):
# From the stock model we know:
#   rob_adopted = total_stock * (1/EUL)                               [100% of ROB population upgraded]
#   ret_er_adopted = total_stock * (1 - 1/EUL) * (1/3) * 0.02        [2% of RET_ER population upgraded]
# 
# Since total_adopted = rob_adopted + ret_er_adopted, we can derive:
#   rob_proportion = (1/EUL) / [(1/EUL) + (1 - 1/EUL) * (1/3) * 0.02]
#   ret_er_proportion = [(1 - 1/EUL) * (1/3) * 0.02] / [(1/EUL) + (1 - 1/EUL) * (1/3) * 0.02]
# 
# This allows us to split total_adopted back into its components without needing total_stock

In [17]:
# this code now makes the adoption results into columns we can join with avoided cost tables
# first we need to convert df_collapsed to have the correct column names
# the columns ret_er_adopt_t_<time> and rob_adopt_t_<time> will be put into rows with the column header Market and values ret_er and rob respectively
# Time stays in column headers for a wide format

reshaped_adoption_results = {}

for name in collapsed_adoption_results.keys():
    df_collapsed = collapsed_adoption_results[name]
    
    # Get the ID columns (everything except the adopt columns and measure_life)
    id_cols = ['competition_group', 'electric_utility', 'gas_utility', 'building_type', 'efficiency_level', 'measure_life_(yrs)']
    
    # Separate ret_er and rob columns
    ret_er_cols = [col for col in df_collapsed.columns if col.startswith('ret_er_adopt_t_')]
    rob_cols = [col for col in df_collapsed.columns if col.startswith('rob_adopt_t_')]
    
    # Create ret_er dataframe with market column
    df_ret_er = df_collapsed[id_cols + ret_er_cols].copy()
    df_ret_er['market'] = 'ret_er'
    # Rename columns to remove ret_er_ prefix (ret_er_adopt_t_2026 -> adopt_t_2026)
    rename_dict = {col: col.replace('ret_er_', '') for col in ret_er_cols}
    df_ret_er = df_ret_er.rename(columns=rename_dict)
    
    # Create rob dataframe with market column
    df_rob = df_collapsed[id_cols + rob_cols].copy()
    df_rob['market'] = 'rob'
    # Rename columns to remove rob_ prefix (rob_adopt_t_2026 -> adopt_t_2026)
    rename_dict = {col: col.replace('rob_', '') for col in rob_cols}
    df_rob = df_rob.rename(columns=rename_dict)
    
    # Combine ret_er and rob
    df_reshaped = pd.concat([df_ret_er, df_rob], ignore_index=True)
    
    # Reorder columns: id columns, market, then all time columns
    time_cols = [col for col in df_reshaped.columns if col.startswith('adopt_t_')]
    df_reshaped = df_reshaped[id_cols + ['market'] + sorted(time_cols)]
    
    reshaped_adoption_results[name] = df_reshaped
    print(f"\n{name.capitalize()} reshaped adoption shape: {df_reshaped.shape}")

# Display sample
print("\nTech reshaped adoption sample:")
display(reshaped_adoption_results['tech'])
print("\nCompetition reshaped adoption sample:")
display(reshaped_adoption_results['competition'])


Tech reshaped adoption shape: (288, 27)

Competition reshaped adoption shape: (288, 27)

Tech reshaped adoption sample:


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs),market,adopt_t_0,adopt_t_1,adopt_t_10,...,adopt_t_18,adopt_t_19,adopt_t_2,adopt_t_3,adopt_t_4,adopt_t_5,adopt_t_6,adopt_t_7,adopt_t_8,adopt_t_9
0,heating_cooling,none,none,multifamily,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,heating_cooling,none,none,multifamily,top10,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,heating_cooling,none,none,multifamily_li,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,heating_cooling,none,none,multifamily_li,top10,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,heating_cooling,none,none,single_family,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
284,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
285,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
286,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Competition reshaped adoption sample:


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs),market,adopt_t_0,adopt_t_1,adopt_t_10,...,adopt_t_18,adopt_t_19,adopt_t_2,adopt_t_3,adopt_t_4,adopt_t_5,adopt_t_6,adopt_t_7,adopt_t_8,adopt_t_9
0,heating_cooling,none,none,multifamily,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,heating_cooling,none,none,multifamily,top10,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,heating_cooling,none,none,multifamily_li,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,heating_cooling,none,none,multifamily_li,top10,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,heating_cooling,none,none,single_family,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
284,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
285,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
286,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# We will join the reshaped adoption results with df_yrs (avoided cost data)
# Join keys: competition_group, electric_utility, gas_utility, building_type, efficiency_level, subgroup, market
# First, let's check what columns are in df_yrs and what we need to add to reshaped_adoption_results

print("df_yrs columns:")
print(df_yrs.columns.tolist())
print(f"\ndf_yrs shape: {df_yrs.shape}")
print("\ndf_yrs sample:")
display(df_yrs.head())

print("\n" + "="*80 + "\n")

print("reshaped_adoption_results['tech'] columns:")
print(reshaped_adoption_results['tech'].columns.tolist())
print(f"\nreshaped_adoption_results['tech'] shape: {reshaped_adoption_results['tech'].shape}")
print("\nreshaped_adoption_results['tech'] sample:")
display(reshaped_adoption_results['tech'])

df_yrs columns:
['condition', 'competition_group', 'subgroup', 'electric_utility', 'gas_utility', 'building_type', 'measure_life_(yrs)', 'initial_count', 'market', 'count', 'measure_name', 'sector', 'program', 'baseline_condition', 'efficient_condition', 'complementary_condition', 'electric_end_use', 'incremental_cost_(usd)', 'annual_oandm_savings_(usd)', 'annual_energy_saved_(kwh)', 'water_savings_(gallons)', 'energy_impact_1', 'energy_impact_1_fuel_type', 'energy_impact_1_units', 'energy_impact_2', 'energy_impact_2_fuel_type', 'energy_impact_2_units', 'energy_impact_3', 'energy_impact_3_fuel_type', 'energy_impact_3_units', 'energy_impact_1_end_use', 'energy_impact_2_end_use', 'energy_impact_3_end_use', 'measure_incremental_cost', 'demand_ratio', 'deferred_replacement_credit_value', 'measure_water_savings', 'measure_electric_energy_savings', 'measure_natural_gas_savings', 'measure_fuel_oil_savings', 'measure_propane_savings', 'measure_gasoline_savings', 'measure_diesel_savings', 'elec

,condition,competition_group,subgroup,electric_utility,gas_utility,building_type,measure_life_(yrs),initial_count,market,count,...,sct_cost,sct_benefit,sct_bcr,rim_cost,rim_benefit,rim_bcr,pct_cost,pct_benefit,pct_bcr,efficiency_level
0,furnace_oil_efficient_residential,heating_cooling,oil_furnace,test_utility_1,test_utility_1,single_family,10,5000.0,ROB,500.0,...,1680.0,3636.660007,2.164679,1172.716654,4138.658036,3.52912,700.0,4682.630596,6.689472,efficient
1,furnace_oil_efficient_residential,heating_cooling,oil_furnace,test_utility_1,test_utility_2,single_family,10,10000.0,ROB,1000.0,...,1680.0,3645.190210,2.169756,1172.716654,4138.658036,3.52912,700.0,4691.160799,6.701658,efficient
2,furnace_oil_efficient_residential,heating_cooling,oil_furnace,test_utility_1,none,single_family,10,27500.0,ROB,2750.0,...,1680.0,3636.660007,2.164679,1172.716654,4138.658036,3.52912,700.0,4682.630596,6.689472,efficient
9,furnace_natural_gas_efficient_residential,heating_cooling,gas_furnace,test_utility_1,test_utility_1,single_family,10,5000.0,ROB,500.0,...,1680.0,3636.660007,2.164679,1172.716654,4138.658036,3.52912,700.0,4682.630596,6.689472,efficient
10,furnace_natural_gas_efficient_residential,heating_cooling,gas_furnace,test_utility_1,test_utility_2,single_family,10,10000.0,ROB,1000.0,...,1680.0,3645.190210,2.169756,1172.716654,4138.658036,3.52912,700.0,4691.160799,6.701658,efficient




reshaped_adoption_results['tech'] columns:
['competition_group', 'electric_utility', 'gas_utility', 'building_type', 'efficiency_level', 'measure_life_(yrs)', 'market', 'adopt_t_0', 'adopt_t_1', 'adopt_t_10', 'adopt_t_11', 'adopt_t_12', 'adopt_t_13', 'adopt_t_14', 'adopt_t_15', 'adopt_t_16', 'adopt_t_17', 'adopt_t_18', 'adopt_t_19', 'adopt_t_2', 'adopt_t_3', 'adopt_t_4', 'adopt_t_5', 'adopt_t_6', 'adopt_t_7', 'adopt_t_8', 'adopt_t_9']

reshaped_adoption_results['tech'] shape: (288, 27)

reshaped_adoption_results['tech'] sample:


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs),market,adopt_t_0,adopt_t_1,adopt_t_10,...,adopt_t_18,adopt_t_19,adopt_t_2,adopt_t_3,adopt_t_4,adopt_t_5,adopt_t_6,adopt_t_7,adopt_t_8,adopt_t_9
0,heating_cooling,none,none,multifamily,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,heating_cooling,none,none,multifamily,top10,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,heating_cooling,none,none,multifamily_li,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,heating_cooling,none,none,multifamily_li,top10,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,heating_cooling,none,none,single_family,efficient,10,ret_er,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
284,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
285,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
286,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,rob,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
# Now join the reshaped adoption results with the avoided cost data from df_yrs
# The join will match on: competition_group, electric_utility, gas_utility, building_type, efficiency_level, market

joined_results = {}

for name in reshaped_adoption_results.keys():
    df_reshaped = reshaped_adoption_results[name].copy()
    
    # FIX 1: Standardize market column to uppercase to match df_yrs
    # df_yrs has 'ROB', 'RET_ER', 'REMAINING' (uppercase)
    # reshaped has 'ret_er', 'rob' (lowercase)
    df_reshaped['market'] = df_reshaped['market'].str.upper()
    
    print(f"\n{name.capitalize()} market values after standardization: {df_reshaped['market'].unique()}")
    
    # Now join with df_yrs to get avoided costs
    # Get relevant columns from df_yrs (exclude time-based adoption columns, keep cost/benefit columns)
    cost_benefit_cols = [col for col in df_yrs.columns if not col.startswith('adopt_t_')]
    df_yrs_costs = df_yrs[cost_benefit_cols].drop_duplicates()
    
    # FIX 2: Filter df_yrs_costs to only include markets that exist in reshaped data (ROB and RET_ER)
    df_yrs_costs = df_yrs_costs[df_yrs_costs['market'].isin(df_reshaped['market'].unique())]
    
    print(f"   df_yrs_costs shape after filtering: {df_yrs_costs.shape}")
    print(f"   df_yrs_costs markets: {df_yrs_costs['market'].unique()}")
    
    # Perform the join
    df_joined = df_reshaped.merge(
        df_yrs_costs,
        on=['competition_group', 'electric_utility', 'gas_utility', 'building_type', 'efficiency_level', 'market'],
        how='left'
    )
    
    joined_results[name] = df_joined
    print(f"\n{name.capitalize()} joined with costs shape: {df_joined.shape}")
    
    # Check for any null values in key cost columns to verify join success
    sample_cost_cols = [col for col in df_joined.columns if 'cost' in col.lower() or 'benefit' in col.lower()][:5]
    if sample_cost_cols:
        null_counts = df_joined[sample_cost_cols].isnull().sum()
        print(f"   Null counts in sample cost columns:\n{null_counts}")
    
    display(df_joined)

# Store for later use
print("\n" + "="*80)
print("Join completed successfully!")


Tech market values after standardization: ['RET_ER' 'ROB']
   df_yrs_costs shape after filtering: (96, 110)
   df_yrs_costs markets: ['ROB' 'RET_ER']

Tech joined with costs shape: (312, 131)
   Null counts in sample cost columns:
incremental_cost_(usd)                          240
measure_incremental_cost                        240
electric_utility_nonmeasure_program_cost        240
natural_gas_utility_nonmeasure_program_costs    240
nonutility_nonmeasure_program_costs             240
dtype: int64


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs)_x,market,adopt_t_0,adopt_t_1,adopt_t_10,...,trc_bcr,sct_cost,sct_benefit,sct_bcr,rim_cost,rim_benefit,rim_bcr,pct_cost,pct_benefit,pct_bcr
0,heating_cooling,none,none,multifamily,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heating_cooling,none,none,multifamily,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,heating_cooling,none,none,multifamily_li,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,heating_cooling,none,none,multifamily_li,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,heating_cooling,none,none,single_family,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Competition market values after standardization: ['RET_ER' 'ROB']
   df_yrs_costs shape after filtering: (96, 110)
   df_yrs_costs markets: ['ROB' 'RET_ER']

Competition joined with costs shape: (312, 131)
   Null counts in sample cost columns:
incremental_cost_(usd)                          240
measure_incremental_cost                        240
electric_utility_nonmeasure_program_cost        240
natural_gas_utility_nonmeasure_program_costs    240
nonutility_nonmeasure_program_costs             240
dtype: int64


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs)_x,market,adopt_t_0,adopt_t_1,adopt_t_10,...,trc_bcr,sct_cost,sct_benefit,sct_bcr,rim_cost,rim_benefit,rim_bcr,pct_cost,pct_benefit,pct_bcr
0,heating_cooling,none,none,multifamily,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heating_cooling,none,none,multifamily,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,heating_cooling,none,none,multifamily_li,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,heating_cooling,none,none,multifamily_li,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,heating_cooling,none,none,single_family,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Join completed successfully!


In [20]:
# Check which rows have nulls and why
print("Analyzing rows with null cost values:")
df_check = joined_results['tech']
null_mask = df_check['measure_incremental_cost'].isnull()

print(f"\nRows with null costs: {null_mask.sum()} out of {len(df_check)}")
print(f"Rows with valid costs: {(~null_mask).sum()}")

print("\nElectric utility distribution in rows with nulls:")
print(df_check[null_mask]['electric_utility'].value_counts())

print("\nElectric utility distribution in rows with valid costs:")
print(df_check[~null_mask]['electric_utility'].value_counts())

print("\nGas utility distribution in rows with nulls:")
print(df_check[null_mask]['gas_utility'].value_counts())

print("\nSample of rows with nulls:")
display(df_check[null_mask][['competition_group', 'electric_utility', 'gas_utility', 'building_type', 'efficiency_level', 'market']].head(10))

Analyzing rows with null cost values:

Rows with null costs: 240 out of 312
Rows with valid costs: 72

Electric utility distribution in rows with nulls:
electric_utility
none              96
test_utility_2    96
test_utility_1    48
Name: count, dtype: int64

Electric utility distribution in rows with valid costs:
electric_utility
test_utility_1    72
Name: count, dtype: int64

Gas utility distribution in rows with nulls:
gas_utility
none              80
test_utility_1    80
test_utility_2    80
Name: count, dtype: int64

Sample of rows with nulls:


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,market
0,heating_cooling,none,none,multifamily,efficient,RET_ER
1,heating_cooling,none,none,multifamily,top10,RET_ER
2,heating_cooling,none,none,multifamily_li,efficient,RET_ER
3,heating_cooling,none,none,multifamily_li,top10,RET_ER
4,heating_cooling,none,none,single_family,efficient,RET_ER
5,heating_cooling,none,none,single_family,top10,RET_ER
6,heating_cooling,none,none,single_family_li,efficient,RET_ER
7,heating_cooling,none,none,single_family_li,top10,RET_ER
8,heating_cooling,none,test_utility_1,multifamily,efficient,RET_ER
9,heating_cooling,none,test_utility_1,multifamily,top10,RET_ER


In [21]:
# now we just multiply the avoided cost columns by the adoption columns for each time period
# For each adopt_t_<year> column, multiply it by all numeric cost/benefit columns
# and create new columns with the format: <metric_name>_<year>

final_results = {}

for name in joined_results.keys():
    df_joined = joined_results[name].copy()
    
    # Identify adoption time columns (adopt_t_2026, adopt_t_2027, etc.)
    adopt_cols = [col for col in df_joined.columns if col.startswith('adopt_t_')]
    
    # Identify avoided cost/benefit columns (all numeric columns that aren't adopt_t or measure_life)
    # Exclude ID columns and other non-metric columns
    id_cols = ['competition_group', 'electric_utility', 'gas_utility', 'building_type', 
               'efficiency_level', 'market', 'subgroup', 'measure_life_(yrs)']
    
    # Get all numeric columns
    numeric_cols = df_joined.select_dtypes(include=[np.number]).columns.tolist()
    
    # Filter to get only cost/benefit metric columns (exclude adopt_t and measure_life)
    cost_benefit_cols = [col for col in numeric_cols 
                         if not col.startswith('adopt_t_') 
                         and col != 'measure_life_(yrs)']
    
    print(f"\n{name.capitalize()} - Cost/benefit columns to multiply:")
    print(cost_benefit_cols[:10], "..." if len(cost_benefit_cols) > 10 else "")
    print(f"Total: {len(cost_benefit_cols)} metrics")
    
    print(f"\n{name.capitalize()} - Time periods:")
    print(adopt_cols)
    
    # Create all new columns at once using a dictionary (more efficient than adding one by one)
    new_cols_dict = {}
    
    # For each time period, multiply adoption by each cost/benefit metric
    for adopt_col in adopt_cols:
        # Extract year from column name (adopt_t_2026 -> 2026)
        year = adopt_col.replace('adopt_t_', '')
        
        # Multiply each cost/benefit column by this adoption column
        for metric_col in cost_benefit_cols:
            new_col_name = f"{metric_col}_{year}"
            new_cols_dict[new_col_name] = df_joined[metric_col] * df_joined[adopt_col]
    
    # Create a new DataFrame from the dictionary and concatenate with original
    new_cols_df = pd.DataFrame(new_cols_dict, index=df_joined.index)
    df_joined = pd.concat([df_joined, new_cols_df], axis=1)
    
    # Store the result
    final_results[name] = df_joined
    
    print(f"\n{name.capitalize()} final shape: {df_joined.shape}")
    print(f"Added {len(cost_benefit_cols) * len(adopt_cols)} new dollar columns")
    
    # Show sample of new columns
    new_cols = [col for col in df_joined.columns if any(col.endswith(f"_{adopt_cols[0].replace('adopt_t_', '')}") for adopt_col in adopt_cols[:1])]
    print(f"\nSample new columns for first year:")
    print(new_cols[:10])
    
    display(df_joined)

print("\n" + "="*80)
print("Dollar calculations completed!")


Tech - Cost/benefit columns to multiply:
['measure_life_(yrs)_x', 'measure_life_(yrs)_y', 'initial_count', 'count', 'complementary_condition', 'incremental_cost_(usd)', 'annual_oandm_savings_(usd)', 'annual_energy_saved_(kwh)', 'water_savings_(gallons)', 'energy_impact_1'] ...
Total: 92 metrics

Tech - Time periods:
['adopt_t_0', 'adopt_t_1', 'adopt_t_10', 'adopt_t_11', 'adopt_t_12', 'adopt_t_13', 'adopt_t_14', 'adopt_t_15', 'adopt_t_16', 'adopt_t_17', 'adopt_t_18', 'adopt_t_19', 'adopt_t_2', 'adopt_t_3', 'adopt_t_4', 'adopt_t_5', 'adopt_t_6', 'adopt_t_7', 'adopt_t_8', 'adopt_t_9']

Tech final shape: (312, 1971)
Added 1840 new dollar columns

Sample new columns for first year:
['adopt_t_0', 'measure_life_(yrs)_x_0', 'measure_life_(yrs)_y_0', 'initial_count_0', 'count_0', 'complementary_condition_0', 'incremental_cost_(usd)_0', 'annual_oandm_savings_(usd)_0', 'annual_energy_saved_(kwh)_0', 'water_savings_(gallons)_0']


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs)_x,market,adopt_t_0,adopt_t_1,adopt_t_10,...,trc_bcr_9,sct_cost_9,sct_benefit_9,sct_bcr_9,rim_cost_9,rim_benefit_9,rim_bcr_9,pct_cost_9,pct_benefit_9,pct_bcr_9
0,heating_cooling,none,none,multifamily,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heating_cooling,none,none,multifamily,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,heating_cooling,none,none,multifamily_li,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,heating_cooling,none,none,multifamily_li,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,heating_cooling,none,none,single_family,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Competition - Cost/benefit columns to multiply:
['measure_life_(yrs)_x', 'measure_life_(yrs)_y', 'initial_count', 'count', 'complementary_condition', 'incremental_cost_(usd)', 'annual_oandm_savings_(usd)', 'annual_energy_saved_(kwh)', 'water_savings_(gallons)', 'energy_impact_1'] ...
Total: 92 metrics

Competition - Time periods:
['adopt_t_0', 'adopt_t_1', 'adopt_t_10', 'adopt_t_11', 'adopt_t_12', 'adopt_t_13', 'adopt_t_14', 'adopt_t_15', 'adopt_t_16', 'adopt_t_17', 'adopt_t_18', 'adopt_t_19', 'adopt_t_2', 'adopt_t_3', 'adopt_t_4', 'adopt_t_5', 'adopt_t_6', 'adopt_t_7', 'adopt_t_8', 'adopt_t_9']

Competition final shape: (312, 1971)
Added 1840 new dollar columns

Sample new columns for first year:
['adopt_t_0', 'measure_life_(yrs)_x_0', 'measure_life_(yrs)_y_0', 'initial_count_0', 'count_0', 'complementary_condition_0', 'incremental_cost_(usd)_0', 'annual_oandm_savings_(usd)_0', 'annual_energy_saved_(kwh)_0', 'water_savings_(gallons)_0']


,competition_group,electric_utility,gas_utility,building_type,efficiency_level,measure_life_(yrs)_x,market,adopt_t_0,adopt_t_1,adopt_t_10,...,trc_bcr_9,sct_cost_9,sct_benefit_9,sct_bcr_9,rim_cost_9,rim_benefit_9,rim_bcr_9,pct_cost_9,pct_benefit_9,pct_bcr_9
0,heating_cooling,none,none,multifamily,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heating_cooling,none,none,multifamily,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,heating_cooling,none,none,multifamily_li,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,heating_cooling,none,none,multifamily_li,top10,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,heating_cooling,none,none,single_family,efficient,10,RET_ER,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,refrigeration,test_utility_2,test_utility_2,multifamily_li,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,refrigeration,test_utility_2,test_utility_2,single_family,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,refrigeration,test_utility_2,test_utility_2,single_family,top10,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,refrigeration,test_utility_2,test_utility_2,single_family_li,efficient,10,ROB,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Dollar calculations completed!


In [22]:
# Save this final results to pickle and CSV files for both tech and competition scenarios
for name in final_results.keys():
    df_final = final_results[name]
    output_pkl = f"050_output/{name}_final_dollar_results.pkl"
    output_csv = f"050_output/{name}_final_dollar_results.csv"
    
    df_final.to_pickle(output_pkl)
    df_final.to_csv(output_csv, index=False)
    output_pkl_2 = f"060_input/{name}_final_dollar_results.pkl"
    output_csv_2 = f"060_input/{name}_final_dollar_results.csv"
    
    df_final.to_pickle(output_pkl_2)
    df_final.to_csv(output_csv_2, index=False)
    
